In [ ]:
import pandas as pd

original_df = pd.read_excel("GRZS.xlsx")
df = original_df.copy(deep=True)
print(df.shape)
print(df.dtypes)

(7439, 11)
PNID                          int64
Datum                datetime64[ns]
Postaja                      object
Št. ponesrečencev             int64
HeliStCiklov                float64
Dejavnost                    object
PoskodbaVrstaNACA           float64
Vzrok                        object
Izkušenost                   object
Okoliščine                   object
Poškodbe                     object
dtype: object


Odstranimo stolpce, ki nas ne zanimajo(PNID, HeliStCiklov)

In [ ]:
df = df.drop(columns=["PNID", "HeliStCiklov"], axis=1)
print(df.dtypes)

Datum                datetime64[ns]
Postaja                      object
Št. ponesrečencev             int64
Dejavnost                    object
PoskodbaVrstaNACA           float64
Vzrok                        object
Izkušenost                   object
Okoliščine                   object
Poškodbe                     object
dtype: object


Odstranimo duplikate vrednosti, saj so nekatere intervencije podvojene

In [ ]:
df = df.drop_duplicates()
print(df.shape)

(6485, 9)


Treba je tudi filtrirati dejavnosti in ločiti tiste, kateri nas ne zanimajo(letenje z zmajem, promet, drugo(naravna ujma)...)

1. Prvo spremenimo stolpec dejavnost, da bo vsaka unikatna vrednost svoj stolpec - uporabimo one hot encoding s get_dummies()

In [ ]:
df = pd.get_dummies(df, columns=['Dejavnost'])
print(df.dtypes)

Datum                                  datetime64[ns]
Postaja                                        object
Št. ponesrečencev                               int64
PoskodbaVrstaNACA                             float64
Vzrok                                          object
                                            ...      
Dejavnost_planinstvo - hoja po poti              bool
Dejavnost_plezanje                               bool
Dejavnost_promet                                 bool
Dejavnost_turno smučanje                         bool
Dejavnost_vodne aktivnosti                       bool
Length: 440, dtype: object


Kar hitro vidimo da imamo zdaj 441 stolpcev, da bi šli vsakega posebej odstranjevat bi bilo nehumano, zato se bomo lotili problema na drug način.

2. Preštejemo koliko vrednosti je posameznega stolpca
3. Odstranimo tiste, ki se ne ponavljajo pogosto
4. Odstranimo neprimerne

In [ ]:
dummy_cols = [col for col in df.columns if col.startswith("Dejavnost_")]

counts = df[dummy_cols].sum().sort_values()
threshold = 25
df = df.drop(columns=counts[counts < threshold].index)

print(df.dtypes)
print(df.shape)
print()

columns_to_drop = [
    "Dejavnost_delo",
    "Dejavnost_letenje z zmajem balonom ali padalom",
    "Dejavnost_promet",
    "Dejavnost_vodne aktivnosti"
]
df = df.drop(columns=columns_to_drop)

# obdrzimo samo vrstice, kjer so dejavnosti ostale
dummy_cols = [col for col in df.columns if col.startswith("Dejavnost_")]
df = df[df[dummy_cols].sum(axis=1) > 0]

print("Nove vrednosti:")
print(df.shape)
print(df.dtypes)





Datum                                                 datetime64[ns]
Postaja                                                       object
Št. ponesrečencev                                              int64
PoskodbaVrstaNACA                                            float64
Vzrok                                                         object
Izkušenost                                                    object
Okoliščine                                                    object
Poškodbe                                                      object
Dejavnost_alpsko smučanje                                       bool
Dejavnost_delo                                                  bool
Dejavnost_druge športne in rekreacijske aktivnosti              bool
Dejavnost_gorsko kolesarjenje                                   bool
Dejavnost_letenje z zmajem balonom ali padalom                  bool
Dejavnost_planinstvo - brezpotje                                bool
Dejavnost_planinstvo - hoja po pot

Za treniranje klasifikatorja bomo potrebovali več značilnic, kot so vremenske razmere tistega dni, prejšnih 2 dni - to lahko dobimo iz datuma
Potrebovali pa bomo tudi lokacijske informacije, kje se je nesreča zgodili, po kateri poti se je šlo, in tako dalje - to pa bomo probali dobiti iz novic.

Pridobivanje novic o reševalnih akcijah:
1. Pridobis vse novice posameznega gorsko reševalnega društva iz njihovega facebooka/spletne strani (posebej python scripta)
2. za vsako intervencijo iz zbirke najdeš novice, ki so bile objavljene +-1teden od datuma intervencije
3. za vsako izbrano novico preveris ce ustreza intervenciji preko opisa
4. dodas novo vrstico z podatki kraju intervencije, poti itd.
5. Uporabis informacije o datumu in kraju da dobiš vremenske podakte za tisti kraj na tisti datum in 2 dni prej

In [ ]:
df.to_excel("CleanedForScrapingNews.xlsx")
unique = df["Postaja"].unique()
unique

array(['GRS CELJE', 'GRS BOHINJ', 'GRS LJUBLJANA', 'GRS ŠKOFJA LOKA',
       'GRS JESENICE', 'GRS RADOVLJICA', 'GRS KRANJ', 'GRS MARIBOR',
       'GRS TRŽIČ', 'GRS JEZERSKO', 'GRS KAMNIK', 'GRS KOROŠKE',
       'GRS RATEČE', 'GRS TOLMIN', 'GRS BOVEC', 'GRS KRANJSKA GORA',
       'GRS MOJSTRANA'], dtype=object)

Zdaj ko imamo zbrano cimvec podatkoc o posameznih intervencijah, je potrebno te ločene podatke, ki se trenutno nahajajo v ločenih xlsx datotekah, zdruziti po društvih in standardizirati, poleg tega jih je treba očistiti

In [ ]:
import glob
import re
import pandas as pd
from dateutil import parser
from google.colab import files
import os

# Slovenian months → numeric
slovene_months = {
    "januar": "01", "februar": "02", "marec": "03", "april": "04",
    "maj": "05", "junij": "06", "julij": "07", "avgust": "08",
    "september": "09", "oktober": "10", "november": "11", "december": "12"
}

# remove weekday names (anything like "torek," etc.)
weekday_regex = r"^(ponedeljek|torek|sreda|četrtek|petek|sobota|nedelja)[,\s]*"


def normalize_slo_date(date_str):
    """Convert Slovenian-style dates to ISO and parse."""
    if pd.isna(date_str):
        return pd.NaT

    s = str(date_str).strip().lower()

    # remove weekday
    s = re.sub(weekday_regex, "", s).strip()

    # Replace Slovenian month names with numbers
    for month_name, month_num in slovene_months.items():
        pattern = r"\b" + month_name + r"\b"
        if re.search(pattern, s):
            s = re.sub(pattern, month_num, s)
            break

    # Try dateutil parser
    try:
        return parser.parse(s, dayfirst=True, fuzzy=True)
    except:
        pass

    return pd.NaT

seznam_facebook_datotek = [
    "facebook_posts_grsbohinj.xlsx",
    "facebook_posts_GRSLjubljana.xlsx",
    "facebook_posts_grsmaribor.xlsx",
    "facebook_posts_GRSRadovljica.xlsx",
    "facebook_posts_grzs.xlsx",
    "facebook_posts_jesenice.xlsx",
    "facebook_posts_jezersko.xlsx",
    "facebook_posts_koroška.xlsx",
    "facebook_posts_kranj.xlsx",
    "facebook_posts_skofja-loka.xlsx",
    "facebook_posts_tržič.xlsx",
    "facebook_posts_grskamnik.xlsx"
]

seznam_website_datotek = [
    "posts_grs_skofjaloka.xlsx",
    "posts_grs-celje.xlsx",
    "posts_grs-kamnik.xlsx",
    "posts_grs-ljubljana.xlsx",
    "posts_grs-maribor.xlsx",
    "posts_grs-radovljica.xlsx",
    "posts_grs-tržič.xlsx",
    "posts_grs_tolmin.xlsx",
]


def izlusci_drustvo(ime):
    ime = ime.lower()
    ime = re.sub(r"(facebook_posts_|posts_)", "", ime)
    ime = ime.replace(".xlsx", "")
    ime = ime.replace("grs", "")
    ime = ime.replace("-", "").replace("_", "")
    return ime.strip()


zdruzeni = {}
dfs = []

for datoteka in seznam_facebook_datotek + seznam_website_datotek:
    drustvo = izlusci_drustvo(datoteka)

    df = pd.read_excel(datoteka)

    # remove entirely empty rows
    df = df.dropna(how="all")

    if "Vsebina" in df.columns:
        # remove rows where "Vsebina" is empty (missing or only whitespace)
        df = df[df["Vsebina"].notna() & (df["Vsebina"].astype(str).str.strip() != "")]

        # ce vsebina containa samo url jo izbrisi
        url_only_pattern = r"^(?:https?://\S+|www\.\S+)(?:\s+(?:https?://\S+|www\.\S+))*$"
        df = df[~df["Vsebina"].str.match(url_only_pattern, na=False)]

    # standardize "Datum" column (if present)
    if "Datum" in df.columns:
        df["Datum"] = df["Datum"].apply(normalize_slo_date)
        df["Datum"] = pd.to_datetime(df["Datum"], errors="coerce")
        df["Datum"] = df["Datum"].apply(
            lambda x: x.tz_localize(None) if isinstance(x, pd.Timestamp) and x.tzinfo else x
        )

    df = df.drop_duplicates()

    # append to grouped dict
    if drustvo not in zdruzeni:
        zdruzeni[drustvo] = df
    else:
        zdruzeni[drustvo] = pd.concat([zdruzeni[drustvo], df], ignore_index=True)

# final clean + save
for drustvo, df in zdruzeni.items():
    df = df.drop_duplicates()

    # re-normalize Datum after concatenation
    if "Datum" in df.columns:
        df["Datum"] = df["Datum"].apply(normalize_slo_date)
        df["Datum"] = pd.to_datetime(df["Datum"], errors="coerce")
        df["Datum"] = df["Datum"].apply(
            lambda x: x.tz_localize(None) if isinstance(x, pd.Timestamp) and x.tzinfo else x
        )

    df.to_excel(f"{drustvo}.xlsx", index=False)
    dfs.append(df)

print("Narejene datoteke: ")
for k in zdruzeni:
    filename = f"{k}.xlsx"
    if os.path.exists(filename):
        files.download(filename)

zdruzenDf = pd.concat(dfs, ignore_index=True)
print(zdruzenDf.shape)


Narejene datoteke: 


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

(4001, 2)


zdaj smo intervencije ustrezno matchali s novicami s pomočjo LLM-a, to se mormo
malo ročno prečistit, nato pa z pomočjo lokacije in datuma dobimo se vremenske podatke ob casu nesrece

In [ ]:
import pandas as pd

seznam_koncnih_podatkov = [
    "Matched_GRS_BOHINJ.xlsx",
    "Matched_GRS_BOVEC.xlsx",
    "Matched_GRS_CELJE.xlsx",
    "Matched_GRS_JESENICE.xlsx",
    "Matched_GRS_JEZERSKO.xlsx",
    "Matched_GRS_KAMNIK.xlsx",
    "Matched_GRS_KOROŠKE.xlsx",
    "Matched_GRS_KRANJ.xlsx",
    "Matched_GRS_KRANJSKA_GORA.xlsx",
    "Matched_GRS_LJUBLJANA.xlsx",
    "Matched_GRS_MARIBOR.xlsx",
    "Matched_GRS_MOJSTRANA.xlsx",
    "Matched_GRS_RADOVLJICA.xlsx",
    "Matched_GRS_TOLMIN.xlsx",
    "Matched_GRS_TRŽIČ.xlsx",
]

dfs = []

for datoteka in seznam_koncnih_podatkov:
    df = pd.read_excel(datoteka)

    df = df.dropna(how="all")
    df = df.dropna(subset=["Temp_tisti_dan"])

    dfs.append(df)

zdruzenDf = pd.concat(dfs, ignore_index=True)
print(zdruzenDf.shape)

(357, 30)
